# RiskLens — Risk Analysis Demo

End-to-end pipeline: data fetch → processing → risk metrics → Monte Carlo simulation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Data Ingestion

In [ ]:
from src.data.fetch import fetch_asset_data
from src.data.process import clean_market_data, add_returns

TICKER = "BTC-USD"

raw = fetch_asset_data(TICKER)
df = clean_market_data(raw)
df = add_returns(df)

returns = df["returns"]
close = df["close"]

print(f"Fetched {len(df)} trading days for {TICKER}")
df.tail()

## 2. Historical Risk Metrics

In [ ]:
from src.analytics.risk_metrics import (
    annualized_volatility,
    rolling_volatility,
    max_drawdown,
    drawdown_series,
    sharpe_ratio,
    sortino_ratio,
    gain_to_pain_ratio,
    best_worst_periods,
    return_statistics,
)

print("=== Risk ===")
print(f"Annualized Volatility: {annualized_volatility(returns):.2%}")
print(f"Max Drawdown:          {max_drawdown(close):.2%}")
print()
print("=== Performance ===")
print(f"Sharpe Ratio:          {sharpe_ratio(returns):.3f}")
print(f"Sortino Ratio:         {sortino_ratio(returns):.3f}")
print(f"Gain-to-Pain Ratio:    {gain_to_pain_ratio(returns):.3f}")
print()
bw = best_worst_periods(returns, window=21)
print(f"=== Best / Worst {bw['window']}-Day Periods ===")
print(f"Best:  {bw['best']:.2%}")
print(f"Worst: {bw['worst']:.2%}")
print()
print("=== Return Distribution ===")
for k, v in return_statistics(returns).items():
    print(f"  {k:20s}: {v:.6f}")

### 2.1 Returns Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(returns, bins=100, kde=True, ax=ax, color="steelblue")
ax.axvline(returns.mean(), color="red", linestyle="--", label=f"Mean: {returns.mean():.4f}")
ax.set_title(f"{TICKER} — Daily Returns Distribution")
ax.set_xlabel("Daily Return")
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Rolling Volatility

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
rol_vol = rolling_volatility(returns, window=21)
ax.plot(rol_vol.index, rol_vol.values, color="darkorange", linewidth=0.8)
ax.set_title(f"{TICKER} — 21-Day Rolling Annualized Volatility")
ax.set_ylabel("Annualized Volatility")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

### 2.3 Drawdown

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(close.index, close.values, color="steelblue", linewidth=0.8)
axes[0].set_title(f"{TICKER} — Close Price")
axes[0].set_ylabel("Price (USD)")

dd = drawdown_series(close)
axes[1].fill_between(dd.index, dd.values, 0, color="crimson", alpha=0.4)
axes[1].set_title(f"{TICKER} — Drawdown (Max: {max_drawdown(close):.2%})")
axes[1].set_ylabel("Drawdown")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

## 3. Monte Carlo Simulation

In [ ]:
from src.analytics.monte_carlo import (
    simulate_paths,
    simulation_summary,
    prob_target,
    scenario_buckets,
)

N_DAYS = 252
N_SIMS = 10_000
CONFIDENCE = 0.95

paths = simulate_paths(close, returns, n_days=N_DAYS, n_simulations=N_SIMS, seed=42)
final_prices = paths.iloc[-1]
initial_price = close.iloc[-1]

summary = simulation_summary(final_prices, initial_price, confidence=CONFIDENCE)

print(f"=== Will I Win or Lose? ===")
print(f"Probability of GAIN:  {summary['prob_gain']:.1%}")
print(f"Probability of LOSS:  {summary['prob_loss']:.1%}")
print()
print(f"=== Price Outlook ===")
print(f"Initial Price:      ${summary['initial_price']:,.2f}")
print(f"Mean Final Price:   ${summary['mean_final_price']:,.2f}")
print(f"Median Final Price: ${summary['median_final_price']:,.2f}")
print(f"Min Final Price:    ${summary['min_final_price']:,.2f}")
print(f"Max Final Price:    ${summary['max_final_price']:,.2f}")
print()
print(f"=== Downside Risk ===")
print(f"VaR ({CONFIDENCE:.0%}):           {summary['var']:.2%}")
print(f"CVaR ({CONFIDENCE:.0%}):          {summary['cvar']:.2%}")
print()
print(f"=== Upside Opportunity ===")
print(f"Prob of +10%:       {prob_target(final_prices, initial_price, 0.10):.1%}")
print(f"Prob of +25%:       {prob_target(final_prices, initial_price, 0.25):.1%}")
print(f"Prob of +50%:       {prob_target(final_prices, initial_price, 0.50):.1%}")
print(f"Prob of +100%:      {prob_target(final_prices, initial_price, 1.00):.1%}")

### 3.1 Scenario Distribution

In [ ]:
scenarios = scenario_buckets(final_prices, initial_price)

labels = ["Severe Loss\n(<-30%)", "Moderate Loss\n(-30% to -10%)", "Flat\n(-10% to +10%)", "Moderate Gain\n(+10% to +30%)", "Strong Gain\n(>+30%)"]
values = list(scenarios.values())
colors = ["#d62728", "#ff7f0e", "#7f7f7f", "#2ca02c", "#1f77b4"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=0.8)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", fontweight="bold")

ax.set_title(f"{TICKER} — Scenario Distribution after {N_DAYS} Days ({N_SIMS:,} simulations)")
ax.set_ylabel("Probability")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.show()

### 3.2 Simulated Price Paths

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sample = paths.iloc[:, :200]
ax.plot(sample, color="steelblue", alpha=0.03, linewidth=0.5)

ax.plot(paths.median(axis=1), color="white", linewidth=1.5, label="Median")
ax.plot(paths.quantile(0.05, axis=1), color="crimson", linewidth=1, linestyle="--", label="5th percentile")
ax.plot(paths.quantile(0.95, axis=1), color="green", linewidth=1, linestyle="--", label="95th percentile")

ax.axhline(initial_price, color="orange", linewidth=1, linestyle=":", label=f"Initial: ${initial_price:,.0f}")
ax.set_title(f"{TICKER} — Monte Carlo Simulated Paths ({N_SIMS:,} simulations, {N_DAYS} days)")
ax.set_xlabel("Trading Day")
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 3.3 Final Price Distribution with VaR / CVaR

In [ ]:
from src.analytics.monte_carlo import compute_var, compute_cvar

var_pct = compute_var(final_prices, initial_price, CONFIDENCE)
cvar_pct = compute_cvar(final_prices, initial_price, CONFIDENCE)
var_price = initial_price * (1 + var_pct)
cvar_price = initial_price * (1 + cvar_pct)

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(final_prices, bins=100, kde=True, ax=ax, color="steelblue", alpha=0.6)
ax.axvline(initial_price, color="orange", linewidth=1.5, linestyle=":", label=f"Initial: ${initial_price:,.0f}")
ax.axvline(var_price, color="crimson", linewidth=1.5, linestyle="--", label=f"VaR {CONFIDENCE:.0%}: ${var_price:,.0f} ({var_pct:.2%})")
ax.axvline(cvar_price, color="darkred", linewidth=1.5, linestyle="-.", label=f"CVaR {CONFIDENCE:.0%}: ${cvar_price:,.0f} ({cvar_pct:.2%})")
ax.set_title(f"{TICKER} — Final Price Distribution after {N_DAYS} Days")
ax.set_xlabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.show()